# Fase 1B — Induction Heads

Reprodução do protocolo de Olsson et al. (2022) para detecção de induction heads em GPT-2 small.

**Referência:** Olsson, C. et al. (2022). *In-context Learning and Induction Heads.* Anthropic.

**Critério de validação:** as heads com maior prefix matching score devem incluir heads nas camadas 1-2 de GPT-2 small, consistentemente reportadas como induction heads na literatura.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

from src.inspector.model_inspector import ModelInspector, InspectorConfig
from src.circuits.induction_utils import (
    induction_scores,
    top_induction_heads,
    make_repeated_tokens,
)

torch.manual_seed(42)

## 1. Carregar modelo

In [ ]:
inspector = ModelInspector(InspectorConfig(model_name='gpt2', device='auto'))
inspector.load()
print(f'Modelo: GPT-2 small | Layers: {inspector.n_layers} | Heads: {inspector.n_heads}')

## 2. Visualizar padrão de atenção em sequência repetida

Antes de calcular os scores, vamos inspecionar visualmente o padrão de atenção
de uma head candidata (L1H0) numa sequência repetida.

In [ ]:
tokens = make_repeated_tokens(inspector, seq_len=10)
_, cache = inspector.run_tokens(tokens)

# Padrão de L1H0 — candidata a induction head em GPT-2 small
pattern_l1h0 = inspector.get_attention_patterns(cache, layer=1)[0]  # [seq, seq]

fig = px.imshow(
    pattern_l1h0.cpu().numpy(),
    title='Atenção L1H0 — sequência repetida [A1..A10 A1..A10]',
    labels={'x': 'Key (token atendido)', 'y': 'Query (token atual)'},
    color_continuous_scale='Blues',
    aspect='auto',
)
fig.show()

## 3. Calcular Prefix Matching Score e Copying Score

Usando 50 sequências aleatórias de comprimento 50.

In [ ]:
N_SEQUENCES = 50
SEQ_LEN = 50

print('Calculando scores (pode levar 1-2 min)...')
scores = induction_scores(inspector, n_sequences=N_SEQUENCES, seq_len=SEQ_LEN)
print('Pronto.')

## 4. Heatmap: Prefix Matching Score

In [ ]:
pm = scores['prefix_matching'].cpu().numpy()

fig = px.imshow(
    pm,
    title='Prefix Matching Score — GPT-2 small',
    labels={'x': 'Head', 'y': 'Layer', 'color': 'Score'},
    x=[f'H{h}' for h in range(inspector.n_heads)],
    y=[f'L{l}' for l in range(inspector.n_layers)],
    color_continuous_scale='RdBu',
    aspect='auto',
)
fig.show()

## 5. Heatmap: Copying Score

In [ ]:
cp = scores['copying'].cpu().numpy()

fig = px.imshow(
    cp,
    title='Copying Score — GPT-2 small',
    labels={'x': 'Head', 'y': 'Layer', 'color': 'Score'},
    x=[f'H{h}' for h in range(inspector.n_heads)],
    y=[f'L{l}' for l in range(inspector.n_layers)],
    color_continuous_scale='RdBu',
    aspect='auto',
)
fig.show()

## 6. Top induction heads detectadas

In [ ]:
top_pm = top_induction_heads(scores, n_top=10, mode='prefix_matching')
top_cp = top_induction_heads(scores, n_top=10, mode='copying')

df_pm = pd.DataFrame(top_pm, columns=['Layer', 'Head', 'PM Score'])
df_pm['Head ID'] = df_pm.apply(lambda r: f'L{int(r.Layer)}H{int(r.Head)}', axis=1)

df_cp = pd.DataFrame(top_cp, columns=['Layer', 'Head', 'CP Score'])
df_cp['Head ID'] = df_cp.apply(lambda r: f'L{int(r.Layer)}H{int(r.Head)}', axis=1)

print('=== Top 10 por Prefix Matching Score ===')
print(df_pm[['Head ID', 'PM Score']].to_string(index=False))
print()
print('=== Top 10 por Copying Score ===')
print(df_cp[['Head ID', 'CP Score']].to_string(index=False))

## 7. Comparação com Olsson et al. (2022)

Induction heads reportadas para GPT-2 small:
- **L1H0, L1H4** (camada 1) — principais induction heads
- Algumas heads na camada 5 também exibem comportamento parcial

In [ ]:
# Heads reportadas em Olsson et al. para GPT-2 small
PAPER_HEADS = {(1, 0), (1, 4)}

detected = {(int(l), int(h)) for l, h, _ in top_pm[:5]}
overlap = PAPER_HEADS & detected

print(f'Heads reportadas no paper:  {sorted(PAPER_HEADS)}')
print(f'Top-5 detectadas (PM score): {sorted(detected)}')
print(f'Sobreposição: {sorted(overlap)}')
print()

# Scores das heads do paper
pm_tensor = scores['prefix_matching']
for (l, h) in sorted(PAPER_HEADS):
    print(f'  L{l}H{h} — PM score: {pm_tensor[l, h].item():.4f}')

## 8. Conclusão — PASS / FAIL

In [ ]:
# Critério: pelo menos 1 das 2 heads do paper está no top-5 por PM score
# E o PM score de L1H0 ou L1H4 é >= 0.1 (acima do ruído)

MIN_SCORE_THRESHOLD = 0.1
paper_scores = [pm_tensor[l, h].item() for l, h in PAPER_HEADS]

condition_overlap = len(overlap) >= 1
condition_score = any(s >= MIN_SCORE_THRESHOLD for s in paper_scores)

if condition_overlap and condition_score:
    print('RESULT: PASS')
    print(f'  Sobreposição com paper: {sorted(overlap)}')
    print(f'  Scores das heads do paper: {[round(s, 4) for s in paper_scores]}')
else:
    print('RESULT: FAIL')
    print(f'  Sobreposição: {sorted(overlap)} (esperado >= 1 head)')
    print(f'  Scores: {[round(s, 4) for s in paper_scores]} (esperado >= {MIN_SCORE_THRESHOLD})')